# Solver Comparison

Compare different optimization methods on the same network.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
from traffic_control import (
    generate_four_junction_example,
    generate_manhattan_example,
        solve_rref,
    solve_lp,
    solve_milp,
    solve_max_flow,
    validate_solution,
    plot_solver_comparison,
)

## Compare on Four Junction Network

In [ ]:
network = generate_four_junction_example()

results = {}
times = {}

for name, solver in [
    ("RREF", lambda n: solve_rref(n)),
    ("LP (min_cost)", lambda n: solve_lp(n, "min_cost")),
    ("LP (max_throughput)", lambda n: solve_lp(n, "max_throughput")),
    ("LP (min_travel_time)", lambda n: solve_lp(n, "min_travel_time")),
    ("LP (balance_load)", lambda n: solve_lp(n, "balance_load")),
    ("MILP", lambda n: solve_milp(n)),
]:
    start = time.perf_counter()
    sol = solver(network)
    elapsed = (time.perf_counter() - start) * 1000
    results[name] = sol
    times[name] = elapsed
    is_valid, _ = validate_solution(network, sol)
    print(f"{name}: {elapsed:.2f}ms, Valid={is_valid}, Obj={sol.objective_value}")

## Visualize Comparison

In [ ]:
fig, ax = plot_solver_comparison(results, figsize=(12, 6))
plt.show()

## Benchmark on Larger Network

In [ ]:
manhattan = generate_manhattan_example()

benchmark_results = {}
for name, solver in [
    ("RREF", lambda n: solve_rref(n)),
    ("LP", lambda n: solve_lp(n)),
]:
    times = []
    for _ in range(10):
        start = time.perf_counter()
        sol = solver(manhattan)
        times.append((time.perf_counter() - start) * 1000)
    benchmark_results[name] = {
        "mean": np.mean(times),
        "std": np.std(times),
        "min": np.min(times),
        "max": np.max(times),
    }
    print(f"{name}: {benchmark_results[name]['mean']:.2f}±{benchmark_results[name]['std']:.2f}ms")

## Flow Comparison Table

In [ ]:
comparison_df = pd.DataFrame({
    method: {k: v.flows.get(k, 0) for k in sorted(set().union(*[s.flows.keys() for s in results.values()]))}
    for method, v in results.items()
}).T
comparison_df.columns = [f"{c} (veh/h)" for c in comparison_df.columns]
comparison_df